# Deep Q-Learning for Flappy Bird

In this notebook we perform a structured study of **Deep Q-Networks (DQN)** applied to the Flappy Bird environment.

DQN is an **off-policy** temporal-difference method that combines:
1. **Function approximation** — a neural network estimates $Q(s, a; \theta)$.
2. **Experience Replay** — past transitions are stored in a buffer and sampled uniformly to break temporal correlations.
3. **Target Network** — a slowly-updated copy $Q(s, a; \theta^{-})$ provides stable TD targets.

### Notebook roadmap
| Part | Description |
|------|-------------|
| **1** | Train a vanilla DQN **without** a target network and observe instability. |
| **2** | Train a full DQN **with** a target network and analyse the improvement. |
| **3** | Side-by-side comparison of both variants (rewards, survival frames). |
| **4** | Greedy evaluation (no exploration) on 50 test episodes. |
| **5** | Visual rendering of the best agent. |

In [2]:
import sys
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from IPython.display import display, clear_output
import torch

# Make sure the project root is on the path
project_root = os.path.abspath('.')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.envs.flappy_bird import make_flappy_bird_env
from src.agents.td_agents_dqn import DQNAgent, DQNReplayBuffer
from src.plotting.plotting import evaluate_flappy_bird_agent

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)

In [3]:

# ======================= GLOBAL CONFIGURATION =======================
# -- Reward shaping --
ALPHA_REWARD = 0.1            # Weight of the proximity-to-gap-centre bonus

# -- Hyperparameters --
LEARNING_RATE = 5e-4
GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.05
NUM_EPISODES = 10_000
EPSILON_DECAY_EPISODES = 8_000
HIDDEN_DIM = 128

# -- Target network --
TARGET_UPDATE_FREQ = 1000     # Hard-copy every N optimisation steps
GRAD_CLIP = 1.0               # Max gradient norm (Huber loss + clipping)

# -- Training modules (toggle on/off) --
TRAIN_NO_TARGET = False
TRAIN_WITH_TARGET = True

# -- Replay buffer --
BUFFER_CAPACITY = 100_000
BATCH_SIZE = 64
MIN_BUFFER_SIZE = 1_000

# -- Evaluation --
NUM_TEST_EPISODES = 50
NUM_RENDER_EPISODES = 3

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)


## Part 1 — Vanilla DQN (No Target Network)

Without a target network the same Q-network is used both to select the greedy action **and** to compute the TD target.
Because the target keeps shifting with every gradient step, training can become unstable and diverge — a phenomenon closely related to the *deadly triad* (function approximation + bootstrapping + off-policy).

$$Q_{\text{target}} = r + \gamma \max_{a'} Q(s', a'; \theta)$$

where $\theta$ is updated at every step, so the target is *non-stationary*.

In [ ]:
env_no_target = make_flappy_bird_env(alpha=ALPHA_REWARD, use_lidar=False)

agent_no_target = DQNAgent(
    env_no_target,
    alpha=LEARNING_RATE,
    gamma=GAMMA,
    epsilon=EPSILON_START,
    hidden_dim=HIDDEN_DIM,
    use_target_network=False,   # <-- no target network
    grad_clip=GRAD_CLIP,
)

buffer_no_target = DQNReplayBuffer(capacity=BUFFER_CAPACITY, state_dim=12)
metrics_no_target = []

if TRAIN_NO_TARGET:
    print(f"Starting DQN (NO target network) training: {NUM_EPISODES} episodes ...")
    start = time.time()

    decay_rate = max(0, (EPSILON_START - EPSILON_END) / EPSILON_DECAY_EPISODES)

    for episode in range(NUM_EPISODES):
        state, _ = env_no_target.reset()
        agent_no_target.epsilon = max(EPSILON_END, EPSILON_START - decay_rate * episode)

        total_reward, steps, total_loss, updates = 0.0, 0, 0.0, 0
        done = False

        while not done:
            action = agent_no_target.get_action(state)
            next_state, reward, terminated, truncated, info = env_no_target.step(action)
            done = terminated or truncated

            buffer_no_target.add(state, action, reward, next_state, done)

            # Train once the buffer has enough transitions
            if len(buffer_no_target) > MIN_BUFFER_SIZE:
                b_s, b_a, b_r, b_ns, b_d = buffer_no_target.sample(BATCH_SIZE)
                loss = agent_no_target.update_batch(b_s, b_a, b_r, b_ns, b_d)
                total_loss += loss
                updates += 1

            state = next_state
            total_reward += reward
            steps += 1

        metrics_no_target.append({
            "episode": episode + 1,
            "reward": total_reward,
            "steps": steps,
            "epsilon": agent_no_target.epsilon,
            "avg_loss": total_loss / updates if updates > 0 else 0,
        })

        if (episode + 1) % 500 == 0:
            m = metrics_no_target[-1]
            print(
                f"[Ep {m['episode']:>5}/{NUM_EPISODES}] "
                f"Reward: {m['reward']:7.1f} | Frames: {m['steps']:4d} | "
                f"Eps: {m['epsilon']:.3f} | Loss: {m['avg_loss']:.4f}"
            )

    agent_no_target.save_weights("models/dqn_no_target_weights.pth")
    df_no_target = pd.DataFrame(metrics_no_target)
    df_no_target.to_csv("results/dqn_no_target_metrics.csv", index=False)
    print(f"\nTraining completed in {time.time() - start:.1f}s.\n")
else:
    print("Loading pre-trained weights (no-target) ...")
    agent_no_target.load_weights("models/dqn_no_target_weights.pth")
    if os.path.exists("results/dqn_no_target_metrics.csv"):
        df_no_target = pd.read_csv("results/dqn_no_target_metrics.csv")
    else:
        df_no_target = pd.DataFrame()



### Learning Curves — Vanilla DQN
Without the target network the reward curve is typically noisy, and the agent may *unlearn* good policies it discovered earlier (catastrophic forgetting amplified by a moving target).


In [ ]:
if not df_no_target.empty:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Cumulative reward
    axes[0].plot(df_no_target["episode"], df_no_target["reward"], alpha=0.25, color="steelblue")
    axes[0].plot(
        df_no_target["episode"],
        df_no_target["reward"].rolling(100, min_periods=10).mean(),
        color="navy", linewidth=2,
    )
    axes[0].set_title("Cumulative Reward")
    axes[0].set_xlabel("Episode")

    # Survival frames
    axes[1].plot(df_no_target["episode"], df_no_target["steps"], alpha=0.25, color="forestgreen")
    axes[1].plot(
        df_no_target["episode"],
        df_no_target["steps"].rolling(100, min_periods=10).mean(),
        color="darkgreen", linewidth=2,
    )
    axes[1].set_title("Survival Frames")
    axes[1].set_xlabel("Episode")

    # Average loss
    axes[2].plot(df_no_target["episode"], df_no_target["avg_loss"], alpha=0.25, color="salmon")
    axes[2].plot(
        df_no_target["episode"],
        df_no_target["avg_loss"].rolling(100, min_periods=10).mean(),
        color="darkred", linewidth=2,
    )
    axes[2].set_title("Average Loss")
    axes[2].set_xlabel("Episode")

    fig.suptitle("DQN without Target Network", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No metrics available for DQN without target network.")


## Part 2 — DQN with Target Network

Introducing a **target network** $Q(s, a; \theta^{-})$ that is synchronised to the online network every `TARGET_UPDATE_FREQ` gradient steps decouples the target from the rapidly changing parameters, greatly stabilising training.

$$Q_{\text{target}} = r + \gamma \max_{a'} Q(s', a'; \theta^{-})$$

The target parameters $\theta^{-}$ are held fixed between synchronisations, yielding a **stationary** regression target for many consecutive updates.

In [ ]:
env_target = make_flappy_bird_env(alpha=ALPHA_REWARD, use_lidar=False)

agent_target = DQNAgent(
    env_target,
    alpha=LEARNING_RATE,
    gamma=GAMMA,
    epsilon=EPSILON_START,
    hidden_dim=HIDDEN_DIM,
    use_target_network=True,   # <-- target network enabled
    target_update_freq=TARGET_UPDATE_FREQ,
    grad_clip=GRAD_CLIP,
)

buffer_target = DQNReplayBuffer(capacity=BUFFER_CAPACITY, state_dim=12)
metrics_target = []

if TRAIN_WITH_TARGET:
    print(f"Starting DQN (WITH target network) training: {NUM_EPISODES} episodes ...")
    start = time.time()

    decay_rate = max(0, (EPSILON_START - EPSILON_END) / EPSILON_DECAY_EPISODES)

    for episode in range(NUM_EPISODES):
        state, _ = env_target.reset()
        agent_target.epsilon = max(EPSILON_END, EPSILON_START - decay_rate * episode)

        total_reward, steps, total_loss, updates = 0.0, 0, 0.0, 0
        done = False

        while not done:
            action = agent_target.get_action(state)
            next_state, reward, terminated, truncated, info = env_target.step(action)
            done = terminated or truncated

            buffer_target.add(state, action, reward, next_state, done)

            if len(buffer_target) > MIN_BUFFER_SIZE:
                b_s, b_a, b_r, b_ns, b_d = buffer_target.sample(BATCH_SIZE)
                loss = agent_target.update_batch(b_s, b_a, b_r, b_ns, b_d)
                total_loss += loss
                updates += 1

            state = next_state
            total_reward += reward
            steps += 1

        metrics_target.append({
            "episode": episode + 1,
            "reward": total_reward,
            "steps": steps,
            "epsilon": agent_target.epsilon,
            "avg_loss": total_loss / updates if updates > 0 else 0,
        })

        if (episode + 1) % 500 == 0:
            m = metrics_target[-1]
            print(
                f"[Ep {m['episode']:>5}/{NUM_EPISODES}] "
                f"Reward: {m['reward']:7.1f} | Frames: {m['steps']:4d} | "
                f"Eps: {m['epsilon']:.3f} | Loss: {m['avg_loss']:.4f}"
            )

    agent_target.save_weights("models/dqn_target_weights.pth")
    df_target = pd.DataFrame(metrics_target)
    df_target.to_csv("results/dqn_target_metrics.csv", index=False)
    print(f"\nTraining completed in {time.time() - start:.1f}s.\n")
else:
    print("Loading pre-trained weights (with target) ...")
    agent_target.load_weights("models/dqn_target_weights.pth")
    if os.path.exists("results/dqn_target_metrics.csv"):
        df_target = pd.read_csv("results/dqn_target_metrics.csv")
    else:
        df_target = pd.DataFrame()


### Learning Curves — DQN with Target Network
The target network should produce smoother, more monotonically improving reward curves compared to the vanilla variant.


In [ ]:

if not df_target.empty:
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    axes[0].plot(df_target["episode"], df_target["reward"], alpha=0.25, color="steelblue")
    axes[0].plot(
        df_target["episode"],
        df_target["reward"].rolling(100, min_periods=10).mean(),
        color="navy", linewidth=2,
    )
    axes[0].set_title("Cumulative Reward")
    axes[0].set_xlabel("Episode")

    axes[1].plot(df_target["episode"], df_target["steps"], alpha=0.25, color="forestgreen")
    axes[1].plot(
        df_target["episode"],
        df_target["steps"].rolling(100, min_periods=10).mean(),
        color="darkgreen", linewidth=2,
    )
    axes[1].set_title("Survival Frames")
    axes[1].set_xlabel("Episode")

    axes[2].plot(df_target["episode"], df_target["avg_loss"], alpha=0.25, color="salmon")
    axes[2].plot(
        df_target["episode"],
        df_target["avg_loss"].rolling(100, min_periods=10).mean(),
        color="darkred", linewidth=2,
    )
    axes[2].set_title("Average Loss")
    axes[2].set_xlabel("Episode")

    fig.suptitle("DQN with Target Network", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No metrics available for DQN with target network.")


## Part 3 — Side-by-Side Comparison

Overlaying the rolling-mean reward and survival curves makes it easy to see how the target network improves learning stability and final performance.


In [ ]:
has_both = not df_no_target.empty and not df_target.empty

if has_both:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))
    window = 100

    # -- Reward comparison --
    ax1.plot(
        df_no_target["episode"],
        df_no_target["reward"].rolling(window, min_periods=10).mean(),
        label="DQN (no target)", linewidth=2, color="coral",
    )
    ax1.plot(
        df_target["episode"],
        df_target["reward"].rolling(window, min_periods=10).mean(),
        label="DQN (target net)", linewidth=2, color="royalblue",
    )
    ax1.set_title("Cumulative Reward (rolling mean)")
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Reward")
    ax1.legend()

    # -- Frames comparison --
    ax2.plot(
        df_no_target["episode"],
        df_no_target["steps"].rolling(window, min_periods=10).mean(),
        label="DQN (no target)", linewidth=2, color="coral",
    )
    ax2.plot(
        df_target["episode"],
        df_target["steps"].rolling(window, min_periods=10).mean(),
        label="DQN (target net)", linewidth=2, color="royalblue",
    )
    ax2.set_title("Survival Frames (rolling mean)")
    ax2.set_xlabel("Episode")
    ax2.set_ylabel("Frames")
    ax2.legend()

    fig.suptitle("Vanilla DQN vs DQN + Target Network", fontsize=15, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Need metrics from both variants to produce a comparison.")


## Part 4 — Greedy Evaluation (Testing)

We evaluate each trained agent for `NUM_TEST_EPISODES` episodes **with $\varepsilon = 0$** (purely greedy).  This gives a fair estimate of the learnt policy quality.


In [ ]:

def evaluate_agent(agent, env, name, num_episodes=NUM_TEST_EPISODES):
    """Run greedy episodes and print summary statistics."""
    original_eps = agent.epsilon
    agent.epsilon = 0.0

    rewards, steps_list, scores = [], [], []
    for _ in range(num_episodes):
        state, _ = env.reset()
        done, total_r, steps, score = False, 0.0, 0, 0
        while not done:
            action = agent.get_action(state)
            state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_r += reward
            steps += 1
            if done:
                score = info.get("score", 0)
        rewards.append(total_r)
        steps_list.append(steps)
        scores.append(score)

    agent.epsilon = original_eps  # restore

    print(f"\n{'=' * 50}")
    print(f"  RESULTS — {name}  ({num_episodes} episodes)")
    print(f"{'=' * 50}")
    print(f"  Avg. reward      : {np.mean(rewards):8.2f} ± {np.std(rewards):6.2f}")
    print(f"  Avg. survival    : {np.mean(steps_list):8.2f} ± {np.std(steps_list):6.2f} frames")
    print(f"  Avg. pipes passed: {np.mean(scores):8.2f} ± {np.std(scores):6.2f}")
    print(f"{'=' * 50}\n")

    return rewards, steps_list, scores

res_no_target = evaluate_agent(agent_no_target, env_no_target, "DQN (no target)")
res_target    = evaluate_agent(agent_target, env_target, "DQN (target net)")


In [ ]:

# Box-plot comparison of test performance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = ["No Target", "Target Net"]
ax1.boxplot([res_no_target[1], res_target[1]], labels=labels)
ax1.set_title("Survival Frames Distribution")
ax1.set_ylabel("Frames")

ax2.boxplot([res_no_target[2], res_target[2]], labels=labels)
ax2.set_title("Pipes Passed Distribution")
ax2.set_ylabel("Pipes")

fig.suptitle("Greedy Evaluation — Test Episodes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()



## Part 5 — Visual Rendering

We render a few episodes of the best model (DQN with target network) to visually inspect the learnt behaviour.


In [ ]:

agent_target.epsilon = 0.0  # Greedy for rendering

try:
    env_vis = make_flappy_bird_env(alpha=ALPHA_REWARD, use_lidar=False, render_mode="human")
except Exception:
    env_vis = make_flappy_bird_env(alpha=ALPHA_REWARD, use_lidar=False, render_mode=None)

evaluate_flappy_bird_agent(
    agent_target, env_vis,
    num_episodes=NUM_RENDER_EPISODES,
    render=(env_vis.render_mode == "human"),
    fps=45,
)

try:
    env_vis.close()
except Exception:
    pass
</VSCode.Cell>
```